In [0]:
%pip install openpyxl

In [0]:
from pathlib import Path

import numpy as np
import pandas as pd

FILE = Path("/Workspace/Users/weimings@andrew.cmu.edu/Datathon-2026/datathon_195_labels_en.xlsx")

# The first two Excel rows are titles; the third row contains column names.
labels = pd.read_excel(FILE, sheet_name="manual_labels", header=2, engine="openpyxl")
features = pd.read_excel(FILE, sheet_name="training_features", header=2, engine="openpyxl")

labels = labels[["row_id", "label"]].dropna(subset=["row_id"]).copy()
features = features.dropna(subset=["row_id"]).copy()

assert set(labels["row_id"]) == set(features["row_id"]), "Row IDs do not match"
df = features.merge(labels, on="row_id", validate="one_to_one")
df["label"] = pd.to_numeric(df["label"], errors="raise")

assert len(df) == 195, f"Expected 195 rows, got {len(df)}"
assert df["label"].isin([0, 1, 2, 3]).all(), "Invalid or missing label"
assert df["query_id"].nunique() == 15
assert df.groupby("query_id").size().eq(13).all()
assert not df.duplicated(["query_id", "facility_id"]).any()

print(f"Ready: {len(df)} rows, {df['query_id'].nunique()} queries")
print(df["label"].value_counts().sort_index())

In [0]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import ndcg_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Use the eight features chosen before evaluating the model.
df["log_distance"] = np.log1p(df["distance_km"])
df["log_direct"] = np.log1p(df["n_direct"])

feature_cols = [
    "log_distance",
    "evidence_score",
    "log_direct",
    "location_weight",
    "is_hospital",
    "issue_count",
    "completeness",
    "capability_bloat",
]

X = df[feature_cols].apply(pd.to_numeric, errors="raise")
y = df["label"].astype(int)
groups = df["query_id"]

def new_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=1.0),
    )

# Current 04_ranking formula for the three routine services in the sample.
df["baseline_score"] = (
    df["evidence_score"]
    * df["location_weight"]
    * np.exp(-df["distance_km"] / 75.0)
)

def evaluate_query(rows, score_col):
    # Break score ties consistently for the top-three list.
    ranked = rows.sort_values(
        [score_col, "distance_km", "facility_id"],
        ascending=[False, True, True],
    )
    return {
        "ndcg_at_5": ndcg_score(
            [rows["label"].to_numpy()],
            [rows[score_col].to_numpy()],
            k=5,
        ),
        "precision_at_3": ranked.head(3)["label"].ge(2).mean(),
    }

results = []
df["ridge_test_score"] = np.nan

for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
    model = new_model()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    test = df.iloc[test_idx].copy()
    test["ridge_test_score"] = model.predict(X.iloc[test_idx])
    df.loc[test.index, "ridge_test_score"] = test["ridge_test_score"]

    for name, col in [
        ("Current formula", "baseline_score"),
        ("Ridge", "ridge_test_score"),
    ]:
        results.append({
            "query_id": test["query_id"].iloc[0],
            "model": name,
            **evaluate_query(test, col),
        })

assert df["ridge_test_score"].notna().all()
per_query = pd.DataFrame(results)
summary = per_query.groupby("model")[["ndcg_at_5", "precision_at_3"]].mean()

print("Average across 15 held-out queries")
display(summary.round(3).reset_index())

# Count how often Ridge improves, ties, or worsens the top-five ranking.
wide = per_query.pivot(index="query_id", columns="model", values="ndcg_at_5")
delta = (wide["Ridge"] - wide["Current formula"]).round(6)
print(f"Ridge wins: {(delta > 0).sum()}, ties: {(delta == 0).sum()}, "
      f"losses: {(delta < 0).sum()}")

In [0]:
final_model = new_model()
final_model.fit(X, y)

ridge = final_model.named_steps["ridge"]
weights = pd.DataFrame({
    "feature": feature_cols,
    "standardized_weight": ridge.coef_,
})
weights["absolute_weight"] = weights["standardized_weight"].abs()
weights = weights.sort_values("absolute_weight", ascending=False)

print(f"Final model trained on {len(df)} labeled candidates")
print(f"Intercept: {ridge.intercept_:.3f}")
display(weights[["feature", "standardized_weight"]].round(3))